# Step 4：原因假设与Step 5：数据验证

## 三个待验证假设

- **H1：**10月7日后大量低留存来源4用户进入，是整体留存下降的主要结构性解释。
- **H2：**来源4用户首周持续使用深度较弱，与低留存表现相关。
- **H3：**来源4用户交易、付费和自动续费结构较弱，说明其订阅质量与其他主要来源存在明显差异。

分析只确认分组差异和描述性结构贡献，不作严格因果表述，也不为来源4编造真实渠道名称。


In [1]:
from pathlib import Path
import time
import numpy as np
import pandas as pd
run_started=time.perf_counter(); project_dir=Path.cwd().resolve()
if project_dir.name=='notebooks': project_dir=project_dir.parent
raw_dir=project_dir/'data'/'raw'; processed_dir=project_dir/'data'/'processed'; CHUNK_SIZE=1_000_000
raw_files=sorted(raw_dir.glob('*.csv')); raw_before={p.name:(p.stat().st_size,p.stat().st_mtime_ns) for p in raw_files}
base=pd.read_csv(processed_dir/'new_user_anomaly_base.csv',parse_dates=['registration_date','first_active_date'],dtype={'msno':'object','registered_via':'Int16'})
base['analysis_period']=np.select([base.registration_date.between('2015-07-01','2015-09-30'),base.registration_date.between('2015-10-01','2015-10-06'),base.registration_date.between('2015-10-07','2015-10-31')],['baseline','oct_01_06','oct_07_31'],default='other')
major_sources=[3,4,7,9]; target=base[base.registration_date.between('2015-10-07','2015-10-31')].copy(); print(f'Base rows={len(base):,}; target users={len(target):,}')

Base rows=628,250; target users=213,113


## H1：来源结构效应复核

In [2]:
source_period=base.groupby(['analysis_period','registered_via'],dropna=False).agg(users=('msno','size'),d7_retention=('d7_retained','mean'),d30_retention=('d30_retained','mean')).reset_index()
source_period['user_share']=source_period.users/source_period.groupby('analysis_period').users.transform('sum')
source_period=source_period[['analysis_period','registered_via','users','user_share','d7_retention','d30_retention']]
baseline=source_period[source_period.analysis_period.eq('baseline')].set_index('registered_via'); october=base[base.registration_month.eq('2015-10')]
oct_mix=october.groupby('registered_via').size()/len(october); baseline_overall=base[base.analysis_period.eq('baseline')].d30_retained.mean(); actual_october=october.d30_retained.mean()
counterfactual=float(sum(oct_mix.get(v,0)*baseline.d30_retention.get(v,0) for v in oct_mix.index)); structure_effect=counterfactual-baseline_overall; within_effect=actual_october-counterfactual
print(source_period.sort_values(['analysis_period','registered_via']).to_string(index=False))
print(f'\nBaseline pooled D30={baseline_overall:.6f}; actual October D30={actual_october:.6f}; counterfactual={counterfactual:.6f}')
print(f'Structure effect={structure_effect:.6f}; within-source effect={within_effect:.6f}; total={actual_october-baseline_overall:.6f}')
source4_late=source_period[(source_period.analysis_period.eq('oct_07_31'))&(source_period.registered_via.eq(4))].iloc[0]
h1_status='支持' if source4_late.user_share>.5 and structure_effect<0 else '不支持'; print(f'H1：{h1_status}')

analysis_period  registered_via  users  user_share  d7_retention  d30_retention
       baseline               2     99    0.000252      0.000000       0.000000
       baseline               3 187274    0.477294      0.253676       0.041842
       baseline               4    475    0.001211      0.021053       0.006316
       baseline               7  56803    0.144770      0.354295       0.319050
       baseline               8     92    0.000234      0.000000       0.000000
       baseline               9 146964    0.374558      0.117485       0.090825
       baseline              11    605    0.001542      0.000000       0.000000
       baseline              14     27    0.000069      0.000000       0.000000
       baseline              16     27    0.000069      0.000000       0.000000
      oct_01_06               2      4    0.000176      0.000000       0.000000
      oct_01_06               3   9975    0.438057      0.266667       0.051028
      oct_01_06               4    134  

## H2：同期主要来源早期激活对比

In [3]:
def q50(s): return s.median()
h2=target[target.registered_via.isin(major_sources)].groupby('registered_via').agg(users=('msno','size'),activated_user_rate=('first_active_date',lambda s:s.notna().mean()),days_to_first_activity_mean=('days_to_first_activity','mean'),days_to_first_activity_median=('days_to_first_activity',q50),first_7d_active_days_mean=('first_7d_active_days','mean'),first_7d_active_days_median=('first_7d_active_days',q50),first_7d_total_secs_mean=('first_7d_total_secs','mean'),first_7d_total_secs_median=('first_7d_total_secs',q50),first_7d_num_100_mean=('first_7d_num_100','mean'),first_7d_num_100_median=('first_7d_num_100',q50),d7_retention=('d7_retained','mean'),d30_retention=('d30_retained','mean')).reset_index()
print(h2.to_string(index=False))
s4=h2[h2.registered_via.eq(4)].iloc[0]; others=h2[h2.registered_via.ne(4)]
lower_behavior=(s4.first_7d_active_days_mean<others.first_7d_active_days_mean.min() and s4.first_7d_total_secs_mean<others.first_7d_total_secs_mean.min() and s4.first_7d_num_100_mean<others.first_7d_num_100_mean.min())
lower_retention=s4.d7_retention<others.d7_retention.min() and s4.d30_retention<others.d30_retention.min(); weaker_activation=s4.activated_user_rate<others.activated_user_rate.min() and s4.days_to_first_activity_mean>others.days_to_first_activity_mean.max(); h2_status='支持' if lower_behavior and lower_retention and weaker_activation else ('部分支持' if lower_behavior and lower_retention else '不支持')
print(f'\nH2：{h2_status}')

 registered_via  users  activated_user_rate  days_to_first_activity_mean  days_to_first_activity_median  first_7d_active_days_mean  first_7d_active_days_median  first_7d_total_secs_mean  first_7d_total_secs_median  first_7d_num_100_mean  first_7d_num_100_median  d7_retention  d30_retention
              3  10804             0.899482                    13.458428                            0.0                   2.306275                          2.0              12805.492156                   2830.3055              43.086264                      7.0      0.149482       0.083117
              4 172259             0.906014                     5.463731                            0.0                   1.988221                          1.0               8306.931549                   1877.8130              27.166337                      4.0      0.050929       0.023708
              7  21142             0.992101                    19.697020                            2.0                   2.515

## 30天交易特征

只使用注册日至注册后30天的交易。相同用户同日多笔交易均计数；`first_*` 选择最早交易日中在原 CSV 最先出现的记录。`revenue_30d` 仅累计 `actual_amount_paid`。

In [4]:
tx_users=target[['msno','registration_date','registered_via']].reset_index(drop=True); n=len(tx_users); user_id=pd.Series(np.arange(n,dtype=np.int32),index=tx_users.msno)
calendar=pd.date_range('2015-01-01','2017-03-31'); ints=calendar.strftime('%Y%m%d').astype('int32'); date_to_ord=pd.Series(np.arange(len(calendar),dtype=np.int16),index=ints); reg_ord=tx_users.registration_date.map(pd.Series(np.arange(len(calendar),dtype=np.int16),index=calendar)).to_numpy('int16')
tx_count=np.zeros(n,dtype=np.int16); paid_count=np.zeros(n,dtype=np.int16); revenue=np.zeros(n,dtype=np.int64); any_auto=np.zeros(n,bool); any_cancel=np.zeros(n,bool)
sentinel=np.iinfo(np.int16).max; first_ord=np.full(n,sentinel,dtype=np.int16); first_plan=np.full(n,-1,dtype=np.int16); first_paid=np.full(n,-1,dtype=np.int32); first_auto=np.full(n,-1,dtype=np.int8)
rows_scanned=qualifying_rows=chunks=0

In [5]:
scan_started=time.perf_counter(); cols=['msno','payment_plan_days','actual_amount_paid','is_auto_renew','transaction_date','is_cancel']
reader=pd.read_csv(raw_dir/'transactions.csv',usecols=cols,dtype={'msno':'object','payment_plan_days':'int16','actual_amount_paid':'int32','is_auto_renew':'int8','transaction_date':'int32','is_cancel':'int8'},chunksize=CHUNK_SIZE)
for chunk in reader:
    rows_scanned+=len(chunk); chunks+=1; mapped=chunk.msno.map(user_id); valid=mapped.notna().to_numpy()
    if valid.any():
        pos=np.flatnonzero(valid); ids=mapped.to_numpy()[valid].astype('int32',copy=False); ords=chunk.transaction_date.iloc[pos].map(date_to_ord).to_numpy(); good_date=~pd.isna(ords)
        ids=ids[good_date]; pos=pos[good_date]; ords=ords[good_date].astype('int16',copy=False); rel=ords.astype('int32')-reg_ord[ids].astype('int32'); keep=(rel>=0)&(rel<=30)
        if keep.any():
            ids=ids[keep]; pos=pos[keep]; ords=ords[keep]; qualifying_rows+=len(ids); amounts=chunk.actual_amount_paid.iloc[pos].to_numpy('int32'); autos=chunk.is_auto_renew.iloc[pos].to_numpy('int8'); cancels=chunk.is_cancel.iloc[pos].to_numpy('int8'); plans=chunk.payment_plan_days.iloc[pos].to_numpy('int16')
            np.add.at(tx_count,ids,1); np.add.at(paid_count,ids,(amounts>0).astype('int16')); np.add.at(revenue,ids,amounts.astype('int64')); np.logical_or.at(any_auto,ids,autos==1); np.logical_or.at(any_cancel,ids,cancels==1)
            cand=pd.DataFrame({'id':ids,'ord':ords,'pos':np.arange(len(ids),dtype=np.int32),'plan':plans,'paid':amounts,'auto':autos}).sort_values(['id','ord','pos']).drop_duplicates('id',keep='first')
            ci=cand.id.to_numpy('int32'); earlier=cand.ord.to_numpy('int16')<first_ord[ci]
            if earlier.any(): ci=ci[earlier]; chosen=cand.iloc[np.flatnonzero(earlier)]; first_ord[ci]=chosen.ord.to_numpy('int16'); first_plan[ci]=chosen.plan.to_numpy('int16'); first_paid[ci]=chosen.paid.to_numpy('int32'); first_auto[ci]=chosen.auto.to_numpy('int8')
    if chunks%10==0: print(f'{rows_scanned:,} transaction rows scanned')
print(f'Scanned {rows_scanned:,} rows in {chunks} chunks; qualifying rows={qualifying_rows:,}; minutes={(time.perf_counter()-scan_started)/60:.2f}')

10,000,000 transaction rows scanned


20,000,000 transaction rows scanned


Scanned 21,547,746 rows in 22 chunks; qualifying rows=44,957; minutes=0.46


In [6]:
has=tx_count>0; first_dates=np.full(n,np.datetime64('NaT'),dtype='datetime64[ns]'); first_dates[has]=calendar.to_numpy()[first_ord[has]]
features=tx_users.copy(); features['has_transaction_30d']=has.astype('int8'); features['transaction_count_30d']=tx_count; features['paid_transaction_count_30d']=paid_count; features['revenue_30d']=revenue
features['first_transaction_date']=pd.to_datetime(first_dates); features['days_to_first_transaction']=(features.first_transaction_date-features.registration_date).dt.days.astype('Int64')
features['first_payment_plan_days']=pd.Series(first_plan).mask(first_plan<0).astype('Int64'); features['first_actual_amount_paid']=pd.Series(first_paid).mask(first_paid<0).astype('Int64'); features['first_is_auto_renew']=pd.Series(first_auto).mask(first_auto<0).astype('Int8')
features['any_auto_renew_30d']=any_auto.astype('int8'); features['any_cancel_30d']=any_cancel.astype('int8')
feature_path=processed_dir/'new_user_transaction_features.csv'; features.to_csv(feature_path,index=False)
assert features.msno.is_unique and len(features)==len(target) and (features.days_to_first_transaction.dropna().between(0,30)).all()
print(f'Rows={len(features):,}; unique users={features.msno.nunique():,}; size={feature_path.stat().st_size:,} bytes'); print(features.columns.tolist())

Rows=213,113; unique users=213,113; size=16,745,236 bytes
['msno', 'registration_date', 'registered_via', 'has_transaction_30d', 'transaction_count_30d', 'paid_transaction_count_30d', 'revenue_30d', 'first_transaction_date', 'days_to_first_transaction', 'first_payment_plan_days', 'first_actual_amount_paid', 'first_is_auto_renew', 'any_auto_renew_30d', 'any_cancel_30d']


## H3：同期主要来源订阅结构

In [7]:
major=features[features.registered_via.isin(major_sources)]
h3=major.groupby('registered_via').agg(users=('msno','size'),has_transaction_30d=('has_transaction_30d','mean'),avg_transaction_count_30d=('transaction_count_30d','mean'),paying_user_rate=('paid_transaction_count_30d',lambda s:(s>0).mean()),avg_revenue_30d=('revenue_30d','mean'),median_revenue_30d=('revenue_30d','median'),first_payment_plan_days_mean=('first_payment_plan_days','mean'),first_payment_plan_days_median=('first_payment_plan_days','median'),first_actual_amount_paid_mean=('first_actual_amount_paid','mean'),first_actual_amount_paid_median=('first_actual_amount_paid','median'),first_is_auto_renew_rate=('first_is_auto_renew','mean'),any_auto_renew_30d=('any_auto_renew_30d','mean'),any_cancel_30d=('any_cancel_30d','mean')).reset_index()
plan_dist=major.dropna(subset=['first_payment_plan_days']).groupby(['registered_via','first_payment_plan_days']).size().rename('users').reset_index(); plan_dist['share_within_source']=plan_dist.users/plan_dist.groupby('registered_via').users.transform('sum')
paid_dist=major.dropna(subset=['first_actual_amount_paid']).groupby(['registered_via','first_actual_amount_paid']).size().rename('users').reset_index(); paid_dist['share_within_source']=paid_dist.users/paid_dist.groupby('registered_via').users.transform('sum')
print('Subscription summary:'); print(h3.to_string(index=False)); print('\nFirst payment_plan_days distribution:'); print(plan_dist.to_string(index=False)); print('\nFirst actual_amount_paid distribution:'); print(paid_dist.to_string(index=False))
s4tx=h3[h3.registered_via.eq(4)].iloc[0]; oth=h3[h3.registered_via.ne(4)]; distinct_tx=(abs(s4tx.has_transaction_30d-oth.has_transaction_30d.mean())>.05 or abs(s4tx.any_auto_renew_30d-oth.any_auto_renew_30d.mean())>.05 or abs(s4tx.avg_revenue_30d-oth.avg_revenue_30d.mean())>10)
h3_status='部分支持' if distinct_tx else '不支持'; print(f'\nH3：{h3_status}')

Subscription summary:
 registered_via  users  has_transaction_30d  avg_transaction_count_30d  paying_user_rate  avg_revenue_30d  median_revenue_30d  first_payment_plan_days_mean  first_payment_plan_days_median  first_actual_amount_paid_mean  first_actual_amount_paid_median  first_is_auto_renew_rate  any_auto_renew_30d  any_cancel_30d
              3  10804             0.128934                   0.142262          0.120233        43.905220                 0.0                      72.01005                            30.0                     320.269921                            149.0                  0.167265            0.021936        0.000370
              4 172259             0.032294                   0.036271          0.030872        10.856820                 0.0                     68.182815                            30.0                     311.490563                            149.0                  0.153155            0.005091        0.000163
              7  21142             0

## 公开数据边界

现有数据可以确认：
- 10月7日前后 `registered_via` 结构发生巨大变化；
- 不同来源的行为和留存差异；
- 不同来源的订阅结构差异。

现有数据不能确认：
- 来源4的真实渠道名称；
- 10月7日是否上线新广告、活动、产品入口；
- `registered_via` 编码是否因业务规则调整而变化。

这些需要结合真实公司的营销、产品和埋点变更日志进一步确认。

In [8]:
raw_after={p.name:(p.stat().st_size,p.stat().st_mtime_ns) for p in raw_files}; raw_unchanged=raw_before==raw_after; runtime=time.perf_counter()-run_started
print(f'Final status: H1={h1_status}; H2={h2_status}; H3={h3_status}')
print(f'Transaction features: {feature_path}; rows={len(features):,}; bytes={feature_path.stat().st_size:,}')
print(f'Total runtime: {runtime:.2f} seconds ({runtime/60:.2f} minutes); raw unchanged={raw_unchanged}')
assert raw_unchanged

Final status: H1=支持; H2=部分支持; H3=部分支持
Transaction features: ./data/processed/new_user_transaction_features.csv; rows=213,113; bytes=16,745,236
Total runtime: 30.62 seconds (0.51 minutes); raw unchanged=True


## 教程式结论：假设状态与结构分解

- **H1：支持。**10月7日至31日来源4用户172,259，占同期新增用户80.83%，来源结构发生剧烈变化。
- **H2：部分支持。**来源4首次活动覆盖率约90.60%，但首周活跃天数、播放时长和完整播放次数较低，且D7/D30较低；体现的是持续使用深度差异，不是“激活失败”。
- **H3：部分支持。**来源4的30天交易率3.23%、付费率3.09%、平均30天收入10.86、自动续费率0.51%，与其他主要来源存在明显订阅结构差异，但不能据此证明其导致留存下降。

### 结构分解（核心亮点）

- 7月至9月 baseline pooled D30：10.02%
- 10月全月 actual D30：7.15%
- 使用10月来源结构与baseline来源留存率得到反事实D30：4.89%
- 来源结构效应：-5.13 pp
- 来源内部表现效应：+2.26 pp
- 净变化：-2.87 pp

**专业解释：**固定各来源baseline留存率，仅替换为10月来源结构，得到反事实D30，从而分离来源结构变化和来源内部表现变化。

**白话解释：**如果每个来源自己的留存表现都不变，仅仅因为10月新增用户来源结构发生变化，整体D30就会被大幅拉低；实际上各来源内部表现反而有所改善，抵消了一部分下降。

**结论：**整体留存下降主要来自新用户来源结构变化，而不是原有主要来源留存普遍恶化。该分解是描述性归因，不是严格因果推断。
